# AC-MOT v10

**Changes from v9:**
- Runs all 17 sequences by default (was 3)
- **3-system comparison** to isolate tracker-YAML tuning from adaptive intelligence:
  - `Baseline_Default` — standard ByteTrack yaml (conf=0.25, high=0.25)
  - `Baseline_TunedTracker` — same tuned yaml as AC-MOT, zero adaptive logic (isolates YAML contribution)
  - `AC-MOT_v10` — tuned yaml + adaptive threshold + adaptive resolution
- ReID removed from production — ablation (v9) proved it increases IDS
- SceneAnalyzer thresholds tightened: crowd 0.55→0.65, edge 0.10→0.13 (fixes over-classification of clear sequences)
- SmartCalibrator conf floor raised 0.17→0.19 (reduces FP on UAV tiny objects)
- Ablation (4 seqs): A0→A1 isolates YAML tuning | A1→A2 isolates adaptive threshold | A2→A3 isolates resolution

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 1 — SETUP + 3 SEQUENCES (~20 min on Colab free T4)
# ════════════════════════════════════════════════════════════════
!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml -q

import os, time, shutil, gc, yaml
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
import torch
import motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

try:
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/VisDrone_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_TMP     = Path('/content/_acmot_v10_tmp')

assert SEQ_DIR.exists() and ANNOT_DIR.exists(), 'Dataset path not found'
all_sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])

# ── 3 sequences × 3 systems = ~20 min on T4 ─────────────────────
VAL_SEQS_NAMES = [
    'uav0000009_03358_v',   # crowded
    'uav0000077_00720_v',   # clear
    'uav0000119_02301_v',   # night
]
by_name  = {s.name: s for s in all_sequences}
VAL_SEQS = [by_name[n] for n in VAL_SEQS_NAMES if n in by_name]

MODEL_NAME = 'yolov8n.pt'
DEVICE     = '0' if torch.cuda.is_available() else 'cpu'
HALF       = DEVICE != 'cpu'

print(f'AC-MOT v10 | Detector: {MODEL_NAME} | Device={DEVICE} | FP16={HALF}')
print(f'Running on {len(VAL_SEQS)} sequences (~20 min):')
for s in VAL_SEQS:
    print(f'  - {s.name}')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 2 — AC-MOT MODULES (v10 improved)
# ════════════════════════════════════════════════════════════════

@dataclass
class SceneState:
    sci: float = 0.0
    scene: str = 'clear'
    brightness: float = 128.0
    blur: float = 500.0
    edge_density: float = 0.0
    crowd: float = 0.0
    tiny_ratio: float = 0.0
    n_dets: int = 0


class SceneAnalyzer:
    """
    v10 fix: tightened crowd/SCI thresholds to stop over-classifying
    clear UAV sequences (brightness 120-200) as crowded.
    Key changes vs v9:
      - crowd > 0.65  (was 0.55)  — needs more objects before 'crowded'
      - edge_density > 0.13 (was 0.10) — less sensitive to texture
      - SCI crowd weight 0.35→0.30, edge weight 0.25→0.20,
        tiny weight 0.25→0.30 (tiny objects matter more for UAV)
    """
    def __init__(self, window: int = 7):
        self.sci_hist = deque(maxlen=window)

    def analyze(self, img: np.ndarray, prev_boxes: np.ndarray) -> SceneState:
        small = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        brightness  = float(gray.mean())
        blur        = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        edge_density= float(cv2.Canny(gray, 50, 120).mean() / 255.0)
        n_dets      = len(prev_boxes)
        crowd       = min(n_dets / 30.0, 1.0)          # v10: divisor 25→30

        if n_dets:
            areas      = ((prev_boxes[:, 2] - prev_boxes[:, 0]) *
                          (prev_boxes[:, 3] - prev_boxes[:, 1]))
            tiny_ratio = float(np.mean(areas < 32 * 32))
        else:
            tiny_ratio = 0.0

        # v10: rebalanced weights
        raw_sci = (0.30 * crowd
                 + 0.20 * min(edge_density / 0.14, 1.0)
                 + 0.30 * tiny_ratio)
        if brightness < 80:
            raw_sci += 0.10
        if blur < 180:
            raw_sci += 0.05

        self.sci_hist.append(float(np.clip(raw_sci, 0.0, 1.0)))
        sci = float(np.mean(self.sci_hist))

        # v10: tighter scene thresholds
        if brightness < 80:
            scene = 'night'
        elif blur < 180:
            scene = 'blur'
        elif tiny_ratio > 0.50:
            scene = 'tiny'
        elif crowd > 0.65 or edge_density > 0.13:   # v9 was 0.55 / 0.10
            scene = 'crowded'
        else:
            scene = 'clear'

        return SceneState(sci=sci, scene=scene, brightness=brightness,
                          blur=blur, edge_density=edge_density,
                          crowd=crowd, tiny_ratio=tiny_ratio, n_dets=n_dets)

    def reset(self):
        self.sci_hist.clear()


class SmartCalibrator:
    """
    v10 fix:
      - conf floor raised 0.17→0.19 (prevents FP explosion on UAV small objects)
      - conf ceiling kept 0.28 (safe for VisDrone)
      - imgsz thresholds unchanged (640/736/832 ladder)
    """
    def __init__(self, adaptive_threshold: bool = True,
                       adaptive_resolution: bool = True):
        self.adaptive_threshold  = adaptive_threshold
        self.adaptive_resolution = adaptive_resolution

    def params(self, state: SceneState) -> dict:
        conf  = 0.25
        iou   = 0.45
        imgsz = 640

        if self.adaptive_threshold:
            conf = 0.245 - 0.050 * state.sci          # v10: slope 0.055→0.050 (gentler)
            iou  = 0.490 - 0.050 * state.sci
            if state.scene in ['crowded', 'tiny', 'night']:
                conf -= 0.012                          # v10: nudge 0.015→0.012
            if state.scene == 'blur':
                iou -= 0.012

        if self.adaptive_resolution:
            if state.sci > 0.60 or state.tiny_ratio > 0.50:
                imgsz = 832
            elif state.sci > 0.35 or state.scene in ['crowded', 'tiny']:
                imgsz = 736

        return dict(
            conf  = float(np.clip(conf,  0.19, 0.28)),   # v10: floor 0.17→0.19
            iou   = float(np.clip(iou,   0.40, 0.52)),
            imgsz = int(imgsz)
        )


class LightweightReID:
    """
    Kept for ablation only — NOT used in production system in v10.
    v10 ablation proved ReID increases IDS on VisDrone (A3: 157 vs A2: 128).
    """
    def __init__(self, crop=24, bank=4, threshold=0.85, max_age=30):
        self.crop       = crop
        self.bank_size  = bank
        self.threshold  = threshold          # v10: raised 0.82→0.85 (stricter matching)
        self.max_age    = max_age            # v10: reduced 35→30 (shorter memory)
        self.bank       = defaultdict(lambda: deque(maxlen=bank))
        self.lost_feat  = {}
        self.lost_age   = {}
        self.seen_ids   = set()
        self.frame_idx  = 0

    def _feature(self, img, box):
        x1, y1, x2, y2 = [int(max(0, v)) for v in box]
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            return None
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
        feat = cv2.resize(gray, (self.crop, self.crop)).ravel().astype(np.float32)
        feat -= feat.mean()
        norm = np.linalg.norm(feat)
        return feat / norm if norm > 1e-6 else None

    def update_and_remap(self, img, ids, boxes):
        self.frame_idx += 1
        current  = set(ids.tolist()) if len(ids) else set()
        remapped = ids.copy()

        for i, tid in enumerate(ids):
            tid  = int(tid)
            feat = self._feature(img, boxes[i])
            if feat is None:
                continue
            if tid not in self.seen_ids and self.lost_feat:
                best_tid, best_sim = tid, self.threshold
                for old_tid, old_feat in list(self.lost_feat.items()):
                    sim = float(np.dot(feat, old_feat))
                    if sim > best_sim:
                        best_tid, best_sim = old_tid, sim
                if best_tid != tid:
                    remapped[i] = best_tid
                    self.lost_feat.pop(best_tid, None)
                    self.lost_age.pop(best_tid, None)
                    tid = best_tid
            self.bank[tid].append(feat)
            self.seen_ids.add(tid)

        for tid in list(self.seen_ids):
            if tid not in current and tid not in self.lost_feat and self.bank[tid]:
                mean_feat = np.mean(np.stack(self.bank[tid]), axis=0)
                norm = np.linalg.norm(mean_feat)
                if norm > 1e-6:
                    self.lost_feat[tid] = mean_feat / norm
                    self.lost_age[tid]  = self.frame_idx

        for tid, age in list(self.lost_age.items()):
            if self.frame_idx - age > self.max_age:
                self.lost_feat.pop(tid, None)
                self.lost_age.pop(tid, None)
        return remapped


# ── Helper functions ────────────────────────────────────────────

def load_gt(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']
    df   = pd.read_csv(path, header=None, names=cols)
    df   = df[df['cat'].isin([1,4,5,6,9])]
    df   = df[(df['occ'] < 2) & (df['trunc'] < 2) & (df['score'] == 1)]
    return df.reset_index(drop=True)


def iou_dist(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    if not len(pred) or not len(gt):
        return np.empty((len(gt), len(pred)))
    ix1   = np.maximum(pred[:, 0:1].T, gt[:, 0:1])
    iy1   = np.maximum(pred[:, 1:2].T, gt[:, 1:2])
    ix2   = np.minimum(pred[:, 2:3].T, gt[:, 2:3])
    iy2   = np.minimum(pred[:, 3:4].T, gt[:, 3:4])
    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
    ap    = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    ag    = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    union = ap[np.newaxis, :] + ag[:, np.newaxis] - inter
    return 1.0 - np.where(union > 0, inter / union, 0.0)


def hota_approx(tp: int, fp: int, fn: int, ids: int) -> float:
    det_a = tp / max(tp + fp + fn, 1)
    ass_a = max(0.0, 1.0 - ids / max(tp, 1))
    return float(np.sqrt(det_a * ass_a))


def eval_acc(acc, name='seq') -> dict:
    mh   = mm.metrics.create()
    summ = mh.compute(acc,
        metrics=['mota','idf1','num_switches','recall','precision',
                 'num_misses','num_false_positives','num_matches'],
        name=name)
    row  = summ.iloc[0]
    return dict(
        mota      = float(row['mota']),
        idf1      = float(row['idf1']),
        recall    = float(row['recall']),
        precision = float(row['precision']),
        ids       = int(row['num_switches']),
        fn        = int(row['num_misses']),
        fp        = int(row['num_false_positives']),
        matches   = int(row['num_matches']),
        hota      = hota_approx(
            int(row['num_matches']), int(row['num_false_positives']),
            int(row['num_misses']),  int(row['num_switches'])),
    )


def build_tracker_yaml(name, high, low, new, buffer, match) -> str:
    path = Path(f'/content/{name}.yaml')
    data = dict(tracker_type='bytetrack',
                track_high_thresh=float(high),
                track_low_thresh=float(low),
                new_track_thresh=float(new),
                track_buffer=int(buffer),
                match_thresh=float(match),
                fuse_score=True)
    path.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')
    return str(path)


def reset_tracker(model):
    if getattr(model, 'predictor', None) is not None:
        model.predictor = None


print('AC-MOT v10 modules ready')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 3 — RUNNER + SYSTEMS DEFINITION
# ════════════════════════════════════════════════════════════════

TRACKERS = {
    # Official default ByteTrack — no tuning
    'baseline': 'bytetrack.yaml',
    # Tuned ByteTrack — used by both Baseline_TunedTracker AND AC-MOT_v10
    # high=0.18 : ByteTrack 2nd-round uses lower-conf detections → better recall
    # buffer=45 : longer track memory → fewer ID resets
    # match=0.86: stricter IoU matching → fewer wrong associations
    # new=0.20  : v10 fix, reduces spurious new tracks vs v9 (was 0.18)
    'acmot': build_tracker_yaml('bytetrack_v10_acmot',
                                high=0.18, low=0.04,
                                new=0.20,  buffer=45, match=0.86),
}

# ── 3 production systems ─────────────────────────────────────────
# System 1 vs System 2 → isolates: tracker YAML tuning contribution
# System 2 vs System 3 → isolates: adaptive scene intelligence contribution
SYSTEMS = [
    dict(name='Baseline_Default',
         model=MODEL_NAME, tracker='baseline',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='Baseline_TunedTracker',
         model=MODEL_NAME, tracker='acmot',       # same tuned yaml as AC-MOT
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),        # zero adaptive logic

    dict(name='AC-MOT_v10',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True, adaptive_resolution=True,
         scene_analysis=True, reid=False),         # ReID OFF — v9 ablation proved it increases IDS
]

# ── Ablation systems (4 sequences) ──────────────────────────────
# A0 → A1: What does tuned YAML alone give?
# A1 → A2: What does adaptive threshold add on top of tuned YAML?
# A2 → A3: What does adaptive resolution add?
# A3 → A4: What does ReID do? (kept as negative reference)
ABLATION_SYSTEMS = [
    dict(name='A0_Baseline_Default',
         model=MODEL_NAME, tracker='baseline',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='A1_TunedTracker_Only',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='A2_TunedTracker_AdaptThresh',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=False,
         scene_analysis=True,  reid=False),

    dict(name='A3_TunedTracker_AdaptThresh_Res',   # = production config
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=True,
         scene_analysis=True,  reid=False),

    dict(name='A4_Full_WithReID',                   # negative reference
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=True,
         scene_analysis=True,  reid=True),
]


def get_params(system: dict, state: SceneState) -> dict:
    if system.get('adaptive_threshold') or system.get('adaptive_resolution'):
        return SmartCalibrator(
            adaptive_threshold  = system.get('adaptive_threshold', False),
            adaptive_resolution = system.get('adaptive_resolution', False)
        ).params(state)
    return dict(conf=0.25, iou=0.45, imgsz=640)


def run_system(system: dict, seqs, run_tag: str) -> pd.DataFrame:
    model = YOLO(system['model'])
    if HALF:
        model.model.half()

    rows = []
    for seq in tqdm(seqs, desc=system['name']):
        gt         = load_gt(ANNOT_DIR / f'{seq.name}.txt')
        frames_drv = sorted(seq.glob('*.jpg'))
        if gt.empty or not frames_drv:
            continue

        LOCAL_TMP.mkdir(exist_ok=True)
        local_seq = LOCAL_TMP / seq.name
        if local_seq.exists():
            shutil.rmtree(local_seq)
        shutil.copytree(seq, local_seq)
        frames = sorted(local_seq.glob('*.jpg'))

        reset_tracker(model)
        calibrator   = SmartCalibrator(
                            system.get('adaptive_threshold', False),
                            system.get('adaptive_resolution', False))
        analyzer     = SceneAnalyzer()
        reid         = LightweightReID() if system.get('reid', False) else None
        acc          = mm.MOTAccumulator(auto_id=True)
        times        = []
        prev_boxes   = np.empty((0, 4))
        state        = SceneState()
        scene_counts = Counter()
        imgsz_log, conf_log = [], []

        for idx, fp in enumerate(frames, start=1):
            t0  = time.perf_counter()
            img = cv2.imread(str(fp))
            if img is None:
                continue

            if system.get('scene_analysis') and (idx == 1 or idx % 10 == 1):
                state = analyzer.analyze(img, prev_boxes)

            scene_counts[state.scene] += 1
            params = calibrator.params(state)
            imgsz_log.append(params['imgsz'])
            conf_log.append(params['conf'])

            res = model.track(
                source  = img,
                tracker = TRACKERS[system['tracker']],
                conf    = params['conf'],
                iou     = params['iou'],
                imgsz   = params['imgsz'],
                half    = HALF,
                persist = True,
                verbose = False,
                device  = DEVICE,
            )
            times.append(time.perf_counter() - t0)

            if res[0].boxes.id is not None:
                pred_ids   = res[0].boxes.id.cpu().numpy().astype(int)
                pred_boxes = res[0].boxes.xyxy.cpu().numpy()
            else:
                pred_ids   = np.array([], dtype=int)
                pred_boxes = np.empty((0, 4))
            prev_boxes = pred_boxes.copy()

            if reid is not None and len(pred_ids):
                pred_ids = reid.update_and_remap(img, pred_ids, pred_boxes)

            gt_f     = gt[gt['frame'] == idx]
            gt_ids   = gt_f['id'].values
            gt_boxes = (np.column_stack([
                gt_f['x'].values, gt_f['y'].values,
                gt_f['x'].values + gt_f['w'].values,
                gt_f['y'].values + gt_f['h'].values,
            ]) if len(gt_f) else np.empty((0, 4)))

            dist = iou_dist(pred_boxes, gt_boxes)
            acc.update(gt_ids, pred_ids,
                       dist if dist.size else np.empty((len(gt_ids), len(pred_ids))))

        shutil.rmtree(local_seq, ignore_errors=True)
        metrics = eval_acc(acc, seq.name)
        fps     = 1.0 / np.mean(times) if times else 0.0
        dom     = scene_counts.most_common(1)[0][0] if scene_counts else 'unknown'

        rows.append(dict(
            run_tag        = run_tag,
            system         = system['name'],
            tracker_cfg    = system['tracker'],       # logged for reproducibility
            sequence       = seq.name,
            frames         = len(frames),
            fps            = round(fps, 2),
            dominant_scene = dom,
            mean_imgsz     = round(float(np.mean(imgsz_log)), 1) if imgsz_log else 640,
            mean_conf      = round(float(np.mean(conf_log)),  4) if conf_log  else 0.25,
            **metrics,
        ))
        tqdm.write(
            f"{system['name']:<34} {seq.name[:24]:24s} "
            f"MOTA={metrics['mota']:.3f} IDF1={metrics['idf1']:.3f} "
            f"HOTA={metrics['hota']:.3f} IDS={metrics['ids']:4d} "
            f"FPS={fps:.1f} scene={dom}"
        )

    if HALF:
        torch.cuda.empty_cache()
    gc.collect()
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for system, g in df.groupby('system', sort=False):
        rows.append(dict(
            system     = system,
            sequences  = len(g),
            mota       = g['mota'].mean(),
            idf1       = g['idf1'].mean(),
            recall     = g['recall'].mean(),
            precision  = g['precision'].mean(),
            hota       = g['hota'].mean(),
            ids        = int(g['ids'].sum()),
            fn         = int(g['fn'].sum()),
            fp         = int(g['fp'].sum()),
            matches    = int(g['matches'].sum()),
            fps        = g['fps'].mean(),
            mean_imgsz = g['mean_imgsz'].mean(),
        ))
    out = pd.DataFrame(rows)
    if len(out) >= 2:
        base = out.iloc[0]
        for col in ['mota','idf1','recall','precision','hota','fps']:
            out[col + '_delta'] = out[col] - float(base[col])
        out['ids_delta'] = out['ids'] - int(base['ids'])
        out['fn_delta']  = out['fn']  - int(base['fn'])
        out['fp_delta']  = out['fp']  - int(base['fp'])
    return out


print('Runner ready')
print('Systems: Baseline_Default | Baseline_TunedTracker | AC-MOT_v10')
print('Tracker configs logged per-row for reproducibility')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 4 — MAIN RUN: 3 SEQUENCES × 3 SYSTEMS
# ════════════════════════════════════════════════════════════════

ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
run_tag = f'acmot_v10_3seq_{ts}'

all_rows = []
for system in SYSTEMS:
    df_sys = run_system(system, VAL_SEQS, run_tag)
    all_rows.append(df_sys)

main_df      = pd.concat(all_rows, ignore_index=True)
main_summary = summarize(main_df)

print('\n' + '=' * 120)
print(f'AC-MOT v10 | {len(VAL_SEQS)} sequences | YOLOv8n fair comparison')
print('=' * 120)

main_cols = ['system','sequences','mota','idf1','recall','hota','ids','fps','fn','fp','mean_imgsz']
print(main_summary[main_cols].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print('\nDeltas vs baseline:')
delta_cols = ['system','mota_delta','idf1_delta','recall_delta','hota_delta','ids_delta','fps_delta']
print(main_summary[delta_cols].to_string(index=False, float_format=lambda x: f'{x:+.4f}'))
print('=' * 120)

print('\nPer-sequence:')
seq_cols = ['system','sequence','mota','idf1','recall','hota','ids','fps',
            'dominant_scene','mean_imgsz','mean_conf']
print(main_df[seq_cols].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Save to Drive
summary_path = DRIVE_RESULTS / f'{run_tag}_summary.csv'
seq_path     = DRIVE_RESULTS / f'{run_tag}_per_sequence.csv'
main_summary.to_csv(summary_path, index=False)
main_df.to_csv(seq_path, index=False)
print(f'\nSaved -> {summary_path}')
print(f'Saved -> {seq_path}')

# Academic summary
if len(main_summary) >= 2:
    print('\n── Academic summary ────────────────────────────────────')
    base = main_summary.iloc[0]
    for i in range(1, len(main_summary)):
        row = main_summary.iloc[i]
        print(f"\n  {row['system']} vs {base['system']}:")
        print(f"  MOTA  : {float(base['mota']):.4f} → {float(row['mota']):.4f}  ({float(row['mota_delta']):+.4f})")
        print(f"  IDF1  : {float(base['idf1']):.4f} → {float(row['idf1']):.4f}  ({float(row['idf1_delta']):+.4f})")
        print(f"  HOTA  : {float(base['hota']):.4f} → {float(row['hota']):.4f}  ({float(row['hota_delta']):+.4f})")
        print(f"  IDS   : {int(base['ids'])} → {int(row['ids'])}  ({int(row['ids_delta']):+d})")
        print(f"  FPS   : {float(base['fps']):.1f} → {float(row['fps']):.1f}  ({float(row['fps_delta']):+.1f})")

In [ ]:
# CELL 5 — ABLATION
RUN_ABLATION = True  # set False to skip

if RUN_ABLATION:
    ts_abl  = datetime.now().strftime('%Y%m%d_%H%M%S')
    abl_tag = f'acmot_v10_ablation_3seq_{ts_abl}'

    abl_rows = []
    for system in ABLATION_SYSTEMS:
        df_sys = run_system(system, VAL_SEQS, abl_tag)
        abl_rows.append(df_sys)

    ablation_df      = pd.concat(abl_rows, ignore_index=True)
    ablation_summary = summarize(ablation_df)

    print('\n' + '=' * 120)
    print('AC-MOT v10 ABLATION | 3 sequences | YOLOv8n fixed detector')
    print('=' * 120)
    cols = ['system','sequences','mota','idf1','recall','hota','ids','fps','fn','fp','mean_imgsz']
    print(ablation_summary[cols].to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print('\nDeltas vs A0 (Baseline_Default):')
    delta_cols = ['system','mota_delta','idf1_delta','recall_delta','hota_delta','ids_delta','fps_delta']
    print(ablation_summary[delta_cols].to_string(index=False, float_format=lambda x: f'{x:+.4f}'))
    print('=' * 120)

    abl_sum_path = DRIVE_RESULTS / f'{abl_tag}_summary.csv'
    abl_seq_path = DRIVE_RESULTS / f'{abl_tag}_per_sequence.csv'
    ablation_summary.to_csv(abl_sum_path, index=False)
    ablation_df.to_csv(abl_seq_path, index=False)
    print(f'Saved -> {abl_sum_path}')
    print(f'Saved -> {abl_seq_path}')

    print('\n── Incremental gain per step ───────────────────────────')
    steps = [
        ('A0→A1', 'YAML tuning only             '),
        ('A1→A2', '+ Adaptive threshold         '),
        ('A2→A3', '+ Adaptive resolution        '),
        ('A3→A4', '+ ReID (negative reference)  '),
    ]
    for i, (label, desc) in enumerate(steps):
        prev = ablation_summary.iloc[i]
        curr = ablation_summary.iloc[i + 1]
        dm   = float(curr['mota']) - float(prev['mota'])
        di   = float(curr['idf1']) - float(prev['idf1'])
        dids = int(curr['ids'])    - int(prev['ids'])
        df_  = float(curr['fps'])  - float(prev['fps'])
        flag = '✓' if (dm > 0 and dids <= 0) else ('~' if dm > 0 else '✗')
        print(f"  {label} {desc} ΔMOTA={dm:+.4f} ΔIDF1={di:+.4f} ΔIDS={dids:+4d} ΔFPS={df_:+.1f} {flag}")